# Embedding.py

This notebook is used to compute the vector embeddings for up to five variants of standardized LOINC lab names. The five variants covered by this notebook include long common name, short name, display name, full name, and consumer name, for a total of ~460k embedded vectors. During testing, we found strong evidence that including consumer name among the embedded vectors substantially reduced model performance (in terms of overall number of nonstandard inputs processed, as well as the standardization accuracy on those inputs). As a result, we have excluded consumer name from our final embeddings, but have left their coverage in this notebook for future experimentation purposes. Excluding consumer name, there are just over 335 thousand embedded vectors across the remaining name variants.

This notebook extracts information about each LOINC code from a tabular file in Azure Blob Storage, formats each piece of data as an input to a `sentence-transformers` model, and then uses CUDA-GPU optimization to rapidly encode them into `pytorch Tensors`. These `Tensors` are then persisted into local memory using a simple `pickle` operation that pairs each vector with its associated LOINC Code, LOINC name string, LOINC Type, and all other corresponding LOINC attributes (such as 'Time Aspect', 'Class Type', 'Properties', etc.).

For optimal performance, a GPU-supported compute instance is **required**. CPU-operations get Tensor-processed at a rate of ~1 iteration/second, while GPU-operations can be processed as fast as 30 iterations/second. We recommend a GPU processor in the A100 series (the best value of speed and power), in particular a standard instance of `NC24ads_A100`. We do not recommend using CPU at all for this task, but if you do, your best bet is simply applying the most processing power you can.

For larger models (e.g. those with many parameters), the notebook also supports mini-batching. This increases the time to compute embeddings but ensures the GPU isn't overloaded and no memory errors occur. For models of up to 8B parameters, mini-batching is not needed, and all vectors can be processed in memory. Any larger, and some mini-batching is required.

Make sure to run this notebook all the way through, including the last cell. The embeddings computation cell leaves a large file in Azure _local_ memory; to make these embeddings broadly available, such as for running the `performance.ipynb` notebook, the file must be moved to Azure Blob Storage. This is handled by the very last cell of this notebook, which also cleans up the local workspace to avoid lingering storage fees.

The resultant embedding files produced by this notebook will be stored in:

`<blob_storage_base>/embeddings/final`

The split-out embedding files (JSONL) will be stored in:

`<blob_storage_base>/embeddings/final/split/<model_name>/`

example: `blob_storage_base/embeddings/final/split/loinc_lab_names_intfloat_e5-large-v2_0.3_1e05_tuned_10000_1e05_20260223`



## Setup

Make sure that once the compute instance is running, you activate the kernel associated with the DIBBs Env in the upper right dropdown. Its packages are correctly optimized for this notebook and avoids some `numpy` instabilities plaguing Azure.

In [ ]:
pip install azure-keyvault-secrets azure-identity azure-ai-ml azureml-fsspec

Now we'll do our basic, standard authentication work. We need all these variables to be able to access our container storage from a file mount. The `DATASTORE_NAME` is not a protected secret and therefore doesn't need to be stashed in KeyVault, as it's a standard Azure default.

In [ ]:
from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential
from azure.keyvault.secrets import SecretClient

# Authenticate to Key Vault
credential = DefaultAzureCredential()
key_vault = "dibbsttc6059789213"
secret_client = SecretClient(vault_url=f"https://{key_vault}.vault.azure.net/", credential=credential)

# Define the workspace subscription and resources so we can instantiate a secure client
SUBSCRIPTION = secret_client.get_secret("subscription").value
RESOURCE_GROUP = secret_client.get_secret("resource-group").value
WS_NAME = secret_client.get_secret("workspace-name").value
DATASTORE_NAME = 'workspaceblobstore'

# NOTE: Even though we're not directly calling any of the client functions, we do still need
# the object. Having a client instantiated acts as an authenticated connection for our compute
# instance to connect to the workspace.
ml_client = MLClient(
    DefaultAzureCredential(),
    SUBSCRIPTION,
    RESOURCE_GROUP,
    WS_NAME,
)

## Step 1: Create File Mount

The Azure Machine Learning File Mount system, though cumbersomely named, allows us to _directly_ access files and objects we have stored in the DIBBs TTC storage container. Any `.txt` or `.csv` files just need to be uploaded to the desired directory within the DIBBs storage container. Unlike with more complex file types, there is no need to manually turn these files into Azure Data Assets before using them. They can simply be directly loaded from storage once they're uploaded.

In [ ]:
from azureml.fsspec import AzureMachineLearningFileSystem

# Instantiate a file system over the workspace so we can interact with data
# assets directly--we get all the goodies like open, ls, etc.
fs = AzureMachineLearningFileSystem(
    f"azureml://subscriptions/{SUBSCRIPTION}/resourcegroups/{RESOURCE_GROUP}/workspaces/{WS_NAME}/datastores/{DATASTORE_NAME}"
)

## Step 2: Load LOINC Code Strings

Using our file mount, we can read the tabular-formatted file of LOINC lab names for processing into two different lists for our transformer model. We decode the bytestrings into proper UTF-8, then combine the four or five names of interest to our work (long common name, short names, display names, full names, and consumer names) into a single list collection (`name_codes`) for the embedding model with a correlated 'details' list collection (`code_details`), that contains the details like LOINC Code, Type, and Axis, that directly corresponds to the specific LOINC code name in the first list.

In [ ]:
SNOINC_CODES_FILE = "./loinc_lab_names_20260223.csv"

print("Extracting SNOINC data to form standardized names...")

name_codes = []
code_details = []
loinc_name_types = []

# NOTE: We found during extensive performance testing that including Consumer Name
# among the embeddings searched as part of Approximate Nearest Neighbor substantially
# reduced model performance (it considered way fewer inputs, and scored way worse on
# them). If you want to include CN in the embeddings after all, change this to `True`.
INCLUDE_CONSUMER_NAME = False

with fs.open(SNOINC_CODES_FILE) as fp:
    # First line is a header giving the column names
    lines_seen = 0
    for line in fp:
        if lines_seen == 0:
            lines_seen += 1
            continue
        
        # Azure Blob Storage is bytes-based, so we need to apply utf decoding
        # before we can use string operations
        line_str = line.decode("utf-8")
        if line_str.strip() != "":
            fields = line_str.strip().split("|")
            # Skip lines that aren't real entries (formatting artifacts)
            # Current order of columns:
            # code|display_name|related_names|definition_desc|lab_type|full_name|property|time_aspect|system|scale_type|method_type|class_type|short_name|long_name|consumer_name

            if len(fields) >= 4:
                code_detail = {
                    "loinc_code": fields[0].strip(),
                    "loinc_type": fields[4].strip(),
                    "axis_property": fields[6].strip(),
                    "axis_time": fields[7].strip(),
                    "axis_system": fields[8].strip(),
                    "axis_scale": fields[9].strip(),
                    "axis_method": fields[10].strip(),
                    "axis_class": fields[11].strip(),
                }
                short_name = fields[12].strip()
                long_common_name = fields[13].strip()
                display_name = fields[1].strip()
                full_name = fields[5].strip()
                if INCLUDE_CONSUMER_NAME:
                    consumer_name = fields[14].strip()

                # Filter out blanks created as a result of formatting/decoding artifacts
                # There should be ~460k loinc names if CN is included, else ~335k
                if short_name != "" and short_name.strip() != "":
                    name_codes.append(short_name)
                    code_details.append(code_detail)
                    loinc_name_types.append("short_name")
                if long_common_name != "" and long_common_name.strip() != "":
                    name_codes.append(long_common_name)
                    code_details.append(code_detail)
                    loinc_name_types.append("long_common_name")
                if display_name != "" and display_name.strip() != "":
                    name_codes.append(display_name)
                    code_details.append(code_detail)
                    loinc_name_types.append("display_name")
                if full_name != "" and full_name.strip() != "":
                    name_codes.append(full_name)
                    code_details.append(code_detail)
                    loinc_name_types.append("full_name")
                if INCLUDE_CONSUMER_NAME:
                    if consumer_name != "" and consumer_name.strip() != "":
                        name_codes.append(consumer_name)
                        code_details.append(code_detail)
                        loinc_name_types.append("consumer_name")

# Whether or not CN is included, there should be at least this many codes unpacked
assert len(name_codes) >= 335_000

print(f"{len(name_codes)} name codes loaded")

## Step 3: Verify GPU Operation

This step may not look like much, but a properly functioning GPU is _imperative_ to the processing capabilities of this script. Pytorch _should_ automatically be able to detect whether the local compute instance is GPU-capable, but when dealing with cloud infrastructure it's best to be sure. We don't even want to try running this if `cuda` isn't accessible.

In [ ]:
import torch
assert torch.cuda.is_available()

## Step 4: Instantiate Language Model

Here, we'll plug in the name of the model we want to embed. If we're loading an "off-the-shelf" model, this name can be found on the Hugging Face page for that model, in the top left, such as [intfloat/e5-large-v2](https://huggingface.co/intfloat/e5-large-v2) here. If we're loading a model we've custome trained and fine-tuned, then we instead need the name of the _model directory_ containing the model's layers and other files (such as `intfloat_e5-larage-v2_0.3_1e05`). When we load the model's properties, we will _explicitly_ toggle it into CUDA-based GPU mode, since we checked above the compute instance can support it.

In [ ]:
import os
from sentence_transformers import SentenceTransformer

# Instantiate the language model
MODEL_NAME = "intfloat_e5-large-v2_0.3_1e05_mnrl_tuned_450000_1e06_no_cn"

# Uncomment the version of the variable that you want, based on the directory the remote model lives in
TRAINED_DIR = "fine_tuned/"
# TRAINED_DIR = "tsdae/"

# First, check if the model exists locally--if it does, nothing to do here
if os.path.exists(MODEL_NAME):
    print("Model exists locally, loading it...")

# If there isn't a local copy, we'll fetch it from remote
else:   
    if fs.exists("models/" + TRAINED_DIR + MODEL_NAME):
        print("Found trained model, loading from remote...")
        fs.get("models/" + TRAINED_DIR + MODEL_NAME, '.')
        print("Model loaded to local memory.")
    else:
        print("Could not find model at specified path.")
        print(
            "Check model name (esp. parameter numbers, underscores, and dashes) and fetch directory."
        )

model = SentenceTransformer(MODEL_NAME, device="cuda")
n_params = sum(p.numel() for p in model.parameters())
print(f"Model uses {n_params} parameters")

## Step 5: Compute Embeddings

This is the bulk of the work of this notebook. Embeddings get computed here, either one-shot or via mini-batching.

For most models the TTC team is working with (see the list below for a full enumeration of all base models), it is likely sufficient to encode all values at once, without needing to rely on mini-batching. We tested models up to 8B parameters in size and encountered no issues during encoding or storage. In this case, set `USE_INCREMENTAL_MINI_BATCHING` to `False`. We also found that a constant `BATCH_SIZE` of 32 was highly performant.

For larger models, or any time the notebook struggles to keep all encodings in memory (due to their size or the number of floating-point operations), you can switch the encoder to mini-batching mode by setting the appropriate boolean to `True`. If mini-batching is used, the `CHUNK_SIZE` parameter defines the size of the mini-batched segment (i.e. how many LOINC code strings are processed before the results are persisted back to disk). We recommend a default value of 8192, but this value can and should be adjusted based on how much the GPU can take before crashing. A higher mini-batching value will enable faster processing, but will put more load on the GPU virtual RAM per batch. If the vector embeddings of a mini-batch can't fit in this VRAM, then the compute instance will crash, and the mini-batch size should be lowered. Just make sure to balance this batch size against the additional time required to repeatedly save and load from disk.

**Troubleshooting Tip:** When handling the large models, or if you have run this notebook several times in sequence, it's possible the compute instance will run out of cache memory. Huggingface downloads the entire backend of every model we instantiate with `sentence-transformers`, which is convenient for our processing but is cumbersome for storage. If you run this cell and are told you're out of memory or the cache is too full, follow these simple steps to clear it:

* Click the three dots next to the compute dropdown and Open Terminal
* Navigate to the compute instance's home directory with `cd ~`
* Run `ls -al` and make sure you can see the directory `.cache` listed. If it's not there, you're not in the right home directory.
* `cd` into `.cache/`. If you run `ls -al`, you should see a folder named `huggingface` or `.huggingface` (depending on settings--which one is present doesn't matter, they're both cache folders).
* Run `rm -rf huggingface/` and wait for the operation to complete. Exit the terminal window (you'll get a pop-up that this will terminate running processes, that's fine). Run the cell again and you're good to go.

The TTC team has tested many models over the course of development. Listing all models and all experimental permutations of training parameters here would bog the notebook down, but a complete listing of all models tested during each phase of development (Initial Model Assessment, TSDAE Domain Adaptation, Fine-Tuning Task Adaptation) can be found in the Team's repo.

**As of March 2026, the DIBBs Text-to-Code team's Final Model is `intfloat_e5-large-v2_0.3_1e05_mnrl_tuned_450000_1e06_no_cn`**.


In [ ]:
import json
import os
from typing import List

import pickle

# This value is always used, regardless of mini-batching mode. Higher batch
# sizes allow the encoder to process more code strings in parallel, but will
# slow down the overall pace of iterations/second. 32 is a very reasonable
# default.
BATCH_SIZE = 32

# This value is only used directly in mini-batching mode.
# However, this value is also used for the splitting of the embedding files into
# JSONL files, if desired, which may slow down the script slightly.
CHUNK_SIZE = 1000

DATE = SNOINC_CODES_FILE.split("_")[-1].split(".")[0]
EMBEDDING_FILE = f"loinc_lab_names_{MODEL_NAME.replace('/', '_')}_{DATE}"

# We'll use the embedding file name to create a local, running-update file which will
# later be 'put' into the embeddings folder in Azure.
SPLIT_OUTPUT_FOLDER = os.path.join(os.getcwd(), "split-embeddings", EMBEDDING_FILE)
os.makedirs(SPLIT_OUTPUT_FOLDER, exist_ok=True)

# NOTE: JSONL split files will be written regardless of whether mini-batching 
# itself is enabled or disabled. It's just that `CHUNK_SIZE` is a parameter 
# re-used between them.
USE_INCREMENTAL_MINI_BATCHING = False

print("Performing embedding, this might take a while...")
if USE_INCREMENTAL_MINI_BATCHING:
    # We'll iterate in chunk-sized intervals to not overload the GPU
    start = 0
    split_id = 0 
    while start < len(name_codes):
        end = min(start + CHUNK_SIZE, len(name_codes))
        mini_batch = name_codes[start:min(end, len(name_codes))]
        mini_batch_loinc_name_types = loinc_name_types[start:min(end, len(loinc_name_types))]
        mini_batch_details = code_details[start:min(end, len(code_details))]

        print("Mini-batching from", start, "to", end - 1)

        # Encoding as a Tensor keeps the result in GPU after calculating it,
        # which allows us to "stack" it on top of the previously computed
        # embeddings in the next step
        batch_embeddings: torch.Tensor = model.encode(
            mini_batch, batch_size=BATCH_SIZE, show_progress_bar=True, convert_to_tensor=True
        )

        try:
            # Not allowed to directly append to a pickle file, so we'll start by
            # reading in all of our previous data.
            with open(EMBEDDING_FILE, "rb") as fp:
                cache_data = pickle.load(fp)
            file_codes: List = cache_data["codes"]
            file_loinc_name_types: List = cache_data["loinc_name_types"]
            saved_embeddings: torch.Tensor = cache_data["embeddings"]
            file_loinc_codes: List = cache_data["loinc_codes"]
            file_loinc_types: List = cache_data["loinc_types"]
            file_properties: List = cache_data["properties"]
            file_times: List = cache_data["time_aspects"]
            file_systems: List = cache_data["systems"]
            file_scales: List = cache_data["scale_types"]
            file_methods: List = cache_data["method_types"]
            file_classes: List = cache_data["class_types"]

            # Then, we'll data-extend it to stack the new mini-batch in. Torch.cat is
            # super efficient so this isn't a problem to re-allocate the objects.
            file_codes.extend(mini_batch)
            file_loinc_name_types.extend(mini_batch_loinc_name_types)
            extended_embeddings = torch.cat((saved_embeddings, batch_embeddings), dim=0)
            file_loinc_codes.extend([d["loinc_code"] for d in mini_batch_details])
            file_loinc_types.extend([d["loinc_type"] for d in mini_batch_details])
            file_properties.extend([d["axis_property"] for d in mini_batch_details])
            file_times.extend([d["axis_time"] for d in mini_batch_details])
            file_systems.extend([d["axis_system"] for d in mini_batch_details])
            file_scales.extend([d["axis_scale"] for d in mini_batch_details])
            file_methods.extend([d["axis_method"] for d in mini_batch_details])
            file_classes.extend([d["axis_class"] for d in mini_batch_details])

            # Finally, we'll overwrite and save it back in-place.
            with open(EMBEDDING_FILE, "wb") as fp:
                pickle.dump({
                    "codes": file_codes,
                    "embeddings": extended_embeddings,
                    "loinc_codes": file_loinc_codes,
                    "loinc_types": file_loinc_types,
                    "loinc_name_types": file_loinc_name_types,
                    "properties": file_properties,
                    "time_aspects": file_times,
                    "systems": file_systems,
                    "scale_types": file_scales,
                    "method_types": file_methods,
                    "class_types": file_classes
                }, fp)

        except FileNotFoundError:
            # File doesn't exist, so just create it
            with open(EMBEDDING_FILE, "wb") as fp:
                pickle.dump({
                    "codes": mini_batch,
                    "embeddings": batch_embeddings,
                    "loinc_codes": [d["loinc_code"] for d in mini_batch_details],
                    "loinc_types": [d["loinc_type"] for d in mini_batch_details],
                    "loinc_name_types": mini_batch_loinc_name_types,
                    "properties": [d["axis_property"] for d in mini_batch_details],
                    "time_aspects": [d["axis_time"] for d in mini_batch_details],
                    "systems": [d["axis_system"] for d in mini_batch_details],
                    "scale_types": [d["axis_scale"] for d in mini_batch_details],
                    "method_types": [d["axis_method"] for d in mini_batch_details],
                    "class_types": [d["axis_class"] for d in mini_batch_details]
                }, fp)

        # Now let's write out the JSONL split embedding file.
        # We'll need to convert to numpy for them to write properly.
        split_embeddings = batch_embeddings.cpu().numpy()
        split_chunk = []

        for i in range(0, len(mini_batch)):
            split_chunk.append(
                {
                    "id": str(split_id),
                    "description": mini_batch[i],
                    "loinc_name_type": mini_batch_loinc_name_types[i],
                    "description_vector": split_embeddings[i].tolist(),
                    "loinc_type": mini_batch_details[i]["loinc_type"],
                    "loinc_code": mini_batch_details[i]["loinc_code"],
                    "property": mini_batch_details[i]["axis_property"],
                    "time_aspect": mini_batch_details[i]["axis_time"],
                    "system": mini_batch_details[i]["axis_system"],
                    "scale_type": mini_batch_details[i]["axis_scale"],
                    "method_type": mini_batch_details[i]["axis_method"],
                    "class_type": mini_batch_details[i]["axis_class"],
                })
            split_id += 1
        with open(f"{SPLIT_OUTPUT_FOLDER}/{EMBEDDING_FILE}_{split_id // CHUNK_SIZE:05d}.jsonl", "w") as f:
            f.writelines(json.dumps(doc) + "\n" for doc in split_chunk)
        start += CHUNK_SIZE

# Regular form of direct processing: no mini-batching here, just bulk-handle all
# the codes at once and write them into a single pickle file.
else:
    corpus_embeddings = model.encode(
        name_codes, batch_size=BATCH_SIZE, show_progress_bar=True, convert_to_tensor=True
    )
    with open(EMBEDDING_FILE, "wb") as fp:
        pickle.dump({
            "codes": name_codes,
            "embeddings": corpus_embeddings,
            "loinc_name_types": loinc_name_types,
            "loinc_codes": [d["loinc_code"] for d in code_details],
            "loinc_types": [d["loinc_type"] for d in code_details],
            "properties": [d["axis_property"] for d in code_details],
            "time_aspects": [d["axis_time"] for d in code_details],
            "systems": [d["axis_system"] for d in code_details],
            "scale_types": [d["axis_scale"] for d in code_details],
            "method_types": [d["axis_method"] for d in code_details],
            "class_types": [d["axis_class"] for d in code_details]
        }, fp)
    
    # Now let's write out the JSONL split embedding file.
    # We'll need to convert to numpy for them to write properly.
    print("Converting embeddings from torch.Tensor to numpy array")
    split_embeddings = corpus_embeddings.cpu().numpy()
    split_chunk_size = 1000

    for i in range(0, len(corpus_embeddings), split_chunk_size):
        split_chunk = [
            {
                "id": str(i + j),
                "description": name_codes[i + j],
                "description_vector": split_embeddings[i + j].tolist(),
                "loinc_type": code_details[i + j]["loinc_type"],
                "loinc_code": code_details[i + j]["loinc_code"],
                "loinc_name_type": loinc_name_types[i + j],
                "property": code_details[i + j]["axis_property"],
                "time_aspect": code_details[i + j]["axis_time"],
                "system": code_details[i + j]["axis_system"],
                "scale_type": code_details[i + j]["axis_scale"],
                "method_type": code_details[i + j]["axis_method"],
                "class_type": code_details[i + j]["axis_class"],
            }
            for j in range(min(split_chunk_size, len(name_codes) - i))
        ]
         
        with open(f"{SPLIT_OUTPUT_FOLDER}/{EMBEDDING_FILE}_{i // split_chunk_size:05d}.jsonl", "w") as f:
            f.writelines(json.dumps(doc) + "\n" for doc in split_chunk)

## Step 6: Cleanup File Locations

With the emebddings computed, we simply move the pickled file into Azure's Blob Storage using our file mount, and then remove the local copy. When this cell completes, if you refresh the notebook pane on the left (the clockwise circular arrow above the dropdown folders), you should see _no_ embedding file in your local workspace. Similarly, in Blob Storage, in the explorer pane, if you refresh there, you _should_ see the embedding file appear.

In [ ]:
import os
import shutil

# Set the first occuring of these variables that is true to True.
# E.g. if this is a fine-tuned model, set FINE_TUNING_USED to True and
# TSDAE_USED to false. If it's not a fine-tuned model but did have TSDAE,
# set the first to False and the second to True. If neither, then set
# both to False.
FINE_TUNING_USED = True
TSDAE_USED = False

# You'll get a directory error if you set both of these to True
assert not (FINE_TUNING_USED and TSDAE_USED)

dir_to_move = "/embeddings/final/"
if FINE_TUNING_USED:
    dir_to_move += "fine_tuned/"
elif TSDAE_USED:
    dir_to_move += "tsdae/"

fs.put(EMBEDDING_FILE, dir_to_move)
print("UPLOADING SPLIT FOLDER TO BLOB STORAGE...")
SPLIT_INPUT_FOLDER = os.path.join(os.getcwd(), "split-embeddings")
fs.upload(lpath=SPLIT_INPUT_FOLDER, rpath="/embeddings/final/split/", recursive=True)

# Cleaning up the split-embeddings folder is a special case, since it 
# contains files. os.rmdir can't handle that, so we need to use
# shutil.rmtree
shutil.rmtree(SPLIT_INPUT_FOLDER)
os.remove(EMBEDDING_FILE)